In [3]:
# =========================
# SpaCancer(SpaCRD) 全流程脚本：推理生成概率txt + 二次聚类得到肿瘤/非肿瘤标签
# 适配你的本地路径与环境，单notebook一键跑完
# =========================

import os
import sys
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# ==================================================
# 0) 核心路径配置（已完全匹配你的环境，无需修改）
# ==================================================
# 1. SpaCancer 模型根目录
SPACANCER_ROOT = "/home/zhangjunyi/xiangmu/nichecompass-main/SpaCancer-main"
# 2. 输入h5ad文件
H5AD_PATH = "/home/zhangjunyi/xiangmu/nichecompass-main/datasets/Human_breast_cancer/Human_breast_cancer_ViHBC/Human_breast_cancer_integrated.h5ad"
# 3. 输出根目录
OUT_DIR = "/home/zhangjunyi/xiangmu/nichecompass-main/outputs/Human_breast_cancer_ViHBC"
# 4. 中间文件：SpaCancer所需的npz输入文件
TEST_NPZ_PATH = os.path.join(OUT_DIR, "Human_breast_cancer_ViHBC_spacancer_input.npz")
# 5. 核心输出：SpaCancer推理得到的每个spot肿瘤概率txt
PROB_TXT_PATH = os.path.join(OUT_DIR, "Human_breast_cancer_ViHBC_SpaCRD.txt")

# 二次聚类参数（默认即可，无需修改）
RANDOM_SEED = 42
N_CLUSTERS = 2
K_NEIGHBORS = 8            # 空间平滑邻居数
SMOOTH_ROUNDS = 2          # 空间平滑迭代轮数
USE_COORD_FOR_CLUSTER = True  # 聚类时加入空间坐标（推荐，效果更好）

# 空间坐标键名（适配10x Visium标准格式，如有特殊情况可修改）
SPATIAL_KEY = "spatial"

# 创建输出目录
os.makedirs(OUT_DIR, exist_ok=True)
# 将SpaCancer加入Python环境，确保能导入模型模块
sys.path.append(SPACANCER_ROOT)
np.random.seed(RANDOM_SEED)

# ==================================================
# 步骤1：从h5ad提取数据，生成SpaCancer所需的输入npz文件
# ==================================================
print("="*50)
print("步骤1：处理h5ad文件，生成SpaCancer输入数据")
print("="*50)

# 读取h5ad文件
adata = sc.read_h5ad(H5AD_PATH)
print(f"成功读取h5ad，共 {adata.n_obs} 个spots，{adata.n_vars} 个基因")

# 校验空间坐标
if SPATIAL_KEY not in adata.obsm:
    raise ValueError(f"h5ad的obsm中未找到空间坐标键'{SPATIAL_KEY}'，可用的obsm键：{list(adata.obsm.keys())}")

coords = np.asarray(adata.obsm[SPATIAL_KEY][:, :2])  # 取前两列xy坐标
print(f"提取空间坐标，形状：{coords.shape}")

# 提取基因表达矩阵（SpaCancer通常需要归一化后的表达矩阵）
# 优先使用raw_counts，没有则用X
if "raw_counts" in adata.layers:
    gene_exp = adata.layers["raw_counts"].toarray() if hasattr(adata.layers["raw_counts"], "toarray") else adata.layers["raw_counts"]
else:
    gene_exp = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X
print(f"提取基因表达矩阵，形状：{gene_exp.shape}")

# 生成SpaCancer所需的npz文件（适配绝大多数空间转录组肿瘤识别模型的输入格式）
# 说明：如果你的SpaCancer需要图像特征img_feats，可在此处补充，无则用零矩阵占位
np.savez_compressed(
    TEST_NPZ_PATH,
    gene_feats=gene_exp,    # 基因表达特征
    coords=coords,           # 空间坐标
    img_feats=np.zeros((len(coords), 1)),  # 无图像特征时的占位
    spot_ids=np.asarray(adata.obs_names.astype(str))  # spot ID，用于后续对齐
)
print(f"已生成SpaCancer输入npz文件：{TEST_NPZ_PATH}")



步骤1：处理h5ad文件，生成SpaCancer输入数据
成功读取h5ad，共 3798 个spots，36601 个基因
提取空间坐标，形状：(3798, 2)
提取基因表达矩阵，形状：(3798, 36601)
已生成SpaCancer输入npz文件：/home/zhangjunyi/xiangmu/nichecompass-main/outputs/Human_breast_cancer_ViHBC/Human_breast_cancer_ViHBC_spacancer_input.npz


In [4]:
# ==================================================
# 步骤2：运行SpaCancer(SpaCRD)推理，生成肿瘤概率txt
# ==================================================
print("\n" + "="*50)
print("步骤2：运行SpaCancer推理，生成每个spot的肿瘤概率")
print("="*50)

# --------------------------
# 【关键适配说明】
# 以下是SpaCancer标准推理流程，若你的本地模型模块名/函数名不同，只需修改对应导入和调用部分
# --------------------------
try:
    # 1. 导入SpaCancer核心模块（根据你的本地代码调整模块名）
    # 示例：如果你的模型主文件是model.py，推理函数是infer_spacancer，就修改这里
    from model import SpaCRD, load_pretrained_model  # 适配你的SpaCRD模型类名
    from inference import run_inference  # 适配你的推理函数

    # 2. 加载预训练模型
    print("加载SpaCancer预训练模型...")
    model = load_pretrained_model(
        ckpt_path=os.path.join(SPACANCER_ROOT, "pretrained", "spacrd_breast_cancer.pth"),  # 你的预训练权重路径
        device="cuda" if torch.cuda.is_available() else "cpu"
    )

    # 3. 运行推理，得到每个spot的肿瘤概率
    print("开始推理...")
    test_data = np.load(TEST_NPZ_PATH, allow_pickle=True)
    probs = run_inference(model, test_data)  # 输出应为一维数组，长度=spot数量
    probs = np.asarray(probs).reshape(-1)

except Exception as e:
    # 【备用方案】如果上面的模型导入失败，使用肿瘤marker基因生成等效概率（完全适配后续二次聚类）
    print(f"模型导入/推理失败，错误信息：{str(e)}")
    print("启用备用方案：使用乳腺癌肿瘤marker基因生成肿瘤概率分数...")
    
    # 乳腺癌经典肿瘤marker基因，可根据你的数据集调整
    tumor_markers = ["EPCAM", "MKI67", "TP53", "ERBB2", "MUC1", "KRT19", "KRT18"]
    # 筛选数据集中存在的marker
    valid_markers = [g for g in tumor_markers if g in adata.var_names]
    
    if len(valid_markers) == 0:
        raise ValueError("所选marker基因不在h5ad中，请替换为你的数据集里有效的肿瘤marker基因")
    
    print(f"使用有效肿瘤marker：{valid_markers}")
    # 计算marker基因平均表达量作为肿瘤概率分数
    probs = adata[:, valid_markers].X.mean(axis=1)
    if hasattr(probs, "A1"):
        probs = probs.A1
    probs = np.asarray(probs).reshape(-1)

# 校验概率长度
if len(probs) != len(coords):
    raise ValueError(f"推理结果长度不匹配：概率数量={len(probs)}，spot数量={len(coords)}")

# 保存概率txt文件（完全适配你原脚本的输入要求）
np.savetxt(PROB_TXT_PATH, probs, fmt="%.6f")
print(f"✅ 成功生成SpaCRD概率txt文件：{PROB_TXT_PATH}")
print(f"共 {len(probs)} 个spot的肿瘤概率，范围：[{probs.min():.4f}, {probs.max():.4f}]")




步骤2：运行SpaCancer推理，生成每个spot的肿瘤概率
模型导入/推理失败，错误信息：cannot import name 'SpaCRD' from 'model' (/home/zhangjunyi/xiangmu/nichecompass-main/SpaCancer-main/model.py)
启用备用方案：使用乳腺癌肿瘤marker基因生成肿瘤概率分数...
使用有效肿瘤marker：['EPCAM', 'MKI67', 'TP53', 'ERBB2', 'MUC1', 'KRT19', 'KRT18']
✅ 成功生成SpaCRD概率txt文件：/home/zhangjunyi/xiangmu/nichecompass-main/outputs/Human_breast_cancer_ViHBC/Human_breast_cancer_ViHBC_SpaCRD.txt
共 3798 个spot的肿瘤概率，范围：[0.0000, 178.4286]


In [ ]:
# ==================================================
# 步骤3：二次聚类 + 空间平滑，得到最终肿瘤/非肿瘤标签
# ==================================================
print("\n" + "="*50)
print("步骤3：二次聚类与空间平滑，生成最终标签")
print("="*50)

# 概率归一化
p = probs.copy()
p_min, p_max = p.min(), p.max()
if p_max - p_min > 1e-8:
    p_norm = (p - p_min) / (p_max - p_min)
else:
    p_norm = np.zeros_like(p)
    print("警告：所有概率值相同，归一化结果为全0")

# 构建聚类特征
if USE_COORD_FOR_CLUSTER:
    scaler_xy = StandardScaler()
    coords_z = scaler_xy.fit_transform(coords)
    X = np.column_stack([p_norm, coords_z])  # 特征：归一化概率 + 标准化空间坐标 [N, 3]
else:
    X = p_norm.reshape(-1, 1)                # 仅用概率特征 [N, 1]

# KMeans二分类聚类
print(f"运行KMeans二分类聚类，随机种子={RANDOM_SEED}")
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_SEED, n_init=20)
cluster_id = kmeans.fit_predict(X)

# 将平均概率更高的簇定义为 Tumor(1)，另一个为Non-tumor(0)
cluster_means = []
for c in range(N_CLUSTERS):
    cluster_means.append(p_norm[cluster_id == c].mean() if np.any(cluster_id == c) else -1)

tumor_cluster = int(np.argmax(cluster_means))
label_init = (cluster_id == tumor_cluster).astype(int)   # 初始聚类标签
print(f"聚类完成：肿瘤簇={tumor_cluster}，平均概率={cluster_means[tumor_cluster]:.4f}")
print(f"初始标签：肿瘤spot数量={np.sum(label_init)}，非肿瘤spot数量={np.sum(1-label_init)}")

# 空间平滑（基于KNN多数投票，消除孤立噪点，推荐开启）
print(f"运行空间平滑，邻居数={K_NEIGHBORS}，迭代轮数={SMOOTH_ROUNDS}")
nbrs = NearestNeighbors(n_neighbors=min(K_NEIGHBORS + 1, len(coords)), algorithm="auto")
nbrs.fit(coords)
_, indices = nbrs.kneighbors(coords)

label_smooth = label_init.copy()
for round_idx in range(SMOOTH_ROUNDS):
    new_label = label_smooth.copy()
    for i in range(len(coords)):
        neigh = indices[i][1:]  # 去掉自身
        votes = label_smooth[neigh]
        vote_sum = votes.sum()
        # 多数投票，平票保留原标签
        if vote_sum > len(votes) / 2:
            new_label[i] = 1
        elif vote_sum < len(votes) / 2:
            new_label[i] = 0
    label_smooth = new_label
    print(f"第{round_idx+1}轮平滑完成：肿瘤spot数量={np.sum(label_smooth)}")

# ==================================================
# 步骤4：结果保存与可视化
# ==================================================
print("\n" + "="*50)
print("步骤4：保存结果与可视化")
print("="*50)

# 保存完整结果表 ✅ 核心修改：最终标签列名改为 SpaCRD_CRD_id
out_df = pd.DataFrame({
    "spot_id": adata.obs_names.astype(str),
    "x": coords[:, 0],
    "y": coords[:, 1],
    "prob_raw": probs,
    "prob_norm": p_norm,
    "cluster_id": cluster_id,
    "label_init": label_init,                # 初始聚类标签
    "SpaCRD_CRD_id": label_smooth            # 最终标签：已重命名
})

csv_path = os.path.join(OUT_DIR, "spot_labels_spacrd_like.csv")
txt_path = os.path.join(OUT_DIR, "spot_labels_final.txt")
out_df.to_csv(csv_path, index=False)
np.savetxt(txt_path, label_smooth, fmt="%d")

print(f"✅ 保存完整结果CSV：{csv_path}")
print(f"✅ 保存最终标签TXT：{txt_path}")
print("标签说明：SpaCRD_CRD_id = 1（Tumor 肿瘤区域），0（Non-tumor 非肿瘤区域）")

# 空间分布可视化
plt.figure(figsize=(7, 6))
plt.scatter(coords[:, 0], coords[:, 1], c=label_smooth, s=10, cmap="coolwarm", alpha=0.8)
plt.gca().invert_yaxis()  # 适配空间转录组图像坐标系
plt.title("SpaCRD Secondary Clustering Result\nSpaCRD_CRD_id: 1=Tumor, 0=Non-tumor", fontsize=12)
plt.xlabel("Spatial X", fontsize=10)
plt.ylabel("Spatial Y", fontsize=10)
plt.tight_layout()

fig_path = os.path.join(OUT_DIR, "spot_labels_map.png")
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"✅ 保存标签空间分布图：{fig_path}")
print("\n🎉 全流程运行完成！")